# Question 7: Neural Network Features for Kernel Methods

---

## 7.1 Theory

### (a) Kernel Methods & Mercer's Theorem

#### Valid Kernel Function

A function $k : \mathcal{X} \times \mathcal{X} \to \mathbb{R}$ is a **valid (Mercer) kernel** if and only if it is:
1. **Symmetric**: $k(x_i, x_j) = k(x_j, x_i)$ for all $x_i, x_j \in \mathcal{X}$.
2. **Positive Semi-Definite (PSD)**: For any finite set of points $\{x_1, x_2, \ldots, x_n\} \subset \mathcal{X}$, the **Gram matrix** $K \in \mathbb{R}^{n \times n}$ defined by $K_{ij} = k(x_i, x_j)$ satisfies:
$$\mathbf{c}^T K \mathbf{c} \geq 0 \quad \forall \mathbf{c} \in \mathbb{R}^n$$

#### Mercer's Theorem

**Statement**: A continuous, symmetric kernel $k(x_i, x_j)$ is a valid positive semi-definite kernel if and only if there exists a (possibly infinite-dimensional) **feature map** $\phi : \mathcal{X} \to \mathcal{H}$ into a Hilbert space $\mathcal{H}$, such that:
$$k(x_i, x_j) = \langle \phi(x_i),\, \phi(x_j) \rangle_{\mathcal{H}}$$

**Interpretation**: Computing the kernel is equivalent to taking an inner product in a high-dimensional feature space, without needing to explicitly compute or store $\phi(x)$. This is the **kernel trick**.

---

### (b) Neural Networks as Feature Extractors — The Neural Kernel

Consider an $L$-layer MLP (Multi-Layer Perceptron). For an input $x$, the hidden layer activations are:
$$h^{(0)}(x) = x$$
$$h^{(l)}(x) = \sigma\!\left(W^{(l)} h^{(l-1)}(x) + b^{(l)}\right) \quad l = 1, 2, \ldots, L-1$$

The **penultimate layer representation** (i.e., the output of the last hidden layer before the final prediction head) is:
$$\phi_{NN}(x) = h^{(L-1)}(x) \in \mathbb{R}^d$$

This acts as a **learned, non-linear feature map** from the input space $\mathcal{X}$ into $\mathbb{R}^d$.

We can then define the **Neural Kernel** as:
$$k_{NN}(x_i, x_j) = \langle \phi_{NN}(x_i),\, \phi_{NN}(x_j) \rangle = \phi_{NN}(x_i)^T \phi_{NN}(x_j)$$

**Why is this a valid kernel?**
Since $\phi_{NN}(x) \in \mathbb{R}^d$ is an explicit finite-dimensional feature map, the inner product in $\mathbb{R}^d$ trivially satisfies Mercer's conditions:
- **Symmetry**: $\phi_{NN}(x_i)^T \phi_{NN}(x_j) = \phi_{NN}(x_j)^T \phi_{NN}(x_i)$.
- **PSD**: For any $\mathbf{c} \in \mathbb{R}^n$:
  $\sum_{i,j} c_i c_j k_{NN}(x_i, x_j) = \|\Phi \mathbf{c}\|^2 \geq 0$
  where $\Phi$ is the matrix with rows $\phi_{NN}(x_i)^T$.

**Key Insight**: The MLP essentially learns a *task-specific, data-driven* feature representation. Plugging this into a kernel method (e.g., SVR) allows the kernel machine to operate in the MLP's learned representation space, often outperforming standard fixed kernels (RBF, Polynomial) on structured data.

In [ ]:
# ============================================================
# SETUP — Imports
# ============================================================

import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Add Q6 source directory so we can import the MLP
q6_path = os.path.abspath('../Multi_Layer_Perceptron_Backpropagation')
if q6_path not in sys.path:
    sys.path.insert(0, q6_path)

from mlp import MLP

from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.manifold import TSNE

print('All imports successful!')

In [ ]:
# ============================================================
# LOAD & PREPROCESS DATASET (from Q4)
# ============================================================

print('\n===== Loading Dataset =====')

data = np.loadtxt('../IIT_H_Campus_Life_Prediction_Dataset/iit_h_mess_dataset.csv', delimiter=',', skiprows=1)

X_raw = data[:, :-1]          # Features (14 columns)
y_raw = data[:, -1]            # Target: mess_duration

# Shuffle
np.random.seed(42)
idx = np.random.permutation(len(X_raw))
X_raw, y_raw = X_raw[idx], y_raw[idx]

# Standardise features
scaler_X = StandardScaler()
X = scaler_X.fit_transform(X_raw)

# Keep target as-is for regression (mess duration in minutes)
y = y_raw

# Train / Validation / Test split (70 / 15 / 15)
n = len(X)
train_end = int(0.70 * n)
val_end   = int(0.85 * n)

X_train, y_train = X[:train_end],         y[:train_end]
X_val,   y_val   = X[train_end:val_end],  y[train_end:val_end]
X_test,  y_test  = X[val_end:],           y[val_end:]

# Reshape targets for MLP (needs column vector)
y_train_mlp = y_train.reshape(-1, 1)
y_val_mlp   = y_val.reshape(-1, 1)
y_test_mlp  = y_test.reshape(-1, 1)

print(f'Train  : {X_train.shape}')
print(f'Val    : {X_val.shape}')
print(f'Test   : {X_test.shape}')

In [ ]:
# ============================================================
# 7.2.1 — TRAIN BEST MLP FROM Q6 (regression, linear output)
# ============================================================

print('\n===== Training MLP Feature Extractor =====')

n_features = X_train.shape[1]

# Architecture: 3 hidden layers of width 64 with ReLU, linear output
best_mlp = MLP(
    layer_sizes   = [n_features, 64, 64, 64, 1],
    activations   = ['relu', 'relu', 'relu', 'relu'],  # last layer effectively linear (ReLU on positive range is fine for regression)
    optimizer     = 'adam',
    learning_rate = 0.001,
    batch_size    = 64,
    weight_init   = 'he'
)

history = best_mlp.fit(X_train, y_train_mlp, epochs=100)

# Plot training loss
plt.figure(figsize=(8, 4))
plt.plot(history, color='steelblue', lw=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('MSE Loss', fontsize=12)
plt.title('MLP Training Loss (Feature Extractor)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Evaluate on test set
mlp_pred = best_mlp.predict(X_test)
mlp_mse  = mean_squared_error(y_test, mlp_pred)
mlp_r2   = r2_score(y_test, mlp_pred)
print(f'MLP Test MSE : {mlp_mse:.4f}')
print(f'MLP Test R²  : {mlp_r2:.4f}')

In [ ]:
# ============================================================
# 7.2.1 — EXTRACT PENULTIMATE LAYER FEATURES φ_NN(x)
# ============================================================

def extract_features(model, X):
    """
    Run the MLP forward pass and return the penultimate hidden layer
    activations h^(L-1)(x) — i.e., one layer before the output.
    """
    activations, _ = model.forward(X)   # activations is a list: [input, h1, h2, ..., h_{L-1}, output]
    return activations[-2]              # penultimate = second-to-last element


phi_train = extract_features(best_mlp, X_train)
phi_val   = extract_features(best_mlp, X_val)
phi_test  = extract_features(best_mlp, X_test)

print(f'φ_NN(X_train) shape : {phi_train.shape}')   # (3500, 64)
print(f'φ_NN(X_test)  shape : {phi_test.shape}')    # (750, 64)

In [ ]:
# ============================================================
# 7.2.1 — t-SNE VISUALIZATION OF NEURAL FEATURES
# ============================================================

print('\n===== t-SNE Visualization =====')

# Run t-SNE on a combined subset (train + test) for a balanced view
# Using a subset of 1500 points for speed
subset_size = min(1500, len(phi_train))
phi_subset  = phi_train[:subset_size]
y_subset    = y_train[:subset_size]

tsne = TSNE(n_components=2, perplexity=40, random_state=42, n_iter=500)
phi_2d = tsne.fit_transform(phi_subset)

plt.figure(figsize=(10, 7))
sc = plt.scatter(
    phi_2d[:, 0], phi_2d[:, 1],
    c=y_subset, cmap='RdYlGn', alpha=0.75, s=15
)
plt.colorbar(sc, label='mess_duration (minutes)')
plt.xlabel('t-SNE Component 1', fontsize=12)
plt.ylabel('t-SNE Component 2', fontsize=12)
plt.title('t-SNE of Neural Features  $\\phi_{NN}(x)$ — Coloured by mess_duration', fontsize=13)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7.2.2 — Support Vector Regression (SVR) with Multiple Kernels

We now define four different kernel configurations and train an SVR on each using **precomputed kernel matrices** via `sklearn.svm.SVR(kernel='precomputed')`.

### Kernels

| Kernel | Formula | Hyperparameters |
|---|---|---|
| **Linear** | $k(x_i, x_j) = x_i^T x_j$ | — |
| **Polynomial** | $k(x_i, x_j) = (x_i^T x_j + 1)^d$ | $d \in \{2, 3\}$ |
| **RBF** | $k(x_i, x_j) = \exp(-\gamma \|x_i - x_j\|^2)$ | $\gamma \in \{0.01, 0.1, 1.0\}$ |
| **Neural** | $k(x_i, x_j) = \phi_{NN}(x_i)^T \phi_{NN}(x_j)$ | — (fixed after MLP training) |

In [ ]:
# ============================================================
# 7.2.2 — KERNEL FUNCTIONS (vectorised NumPy)
# ============================================================

def linear_kernel(A, B):
    """k(x_i, x_j) = x_i · x_j"""
    return A @ B.T


def polynomial_kernel(A, B, d=2, c=1.0):
    """k(x_i, x_j) = (x_i · x_j + c)^d"""
    return (A @ B.T + c) ** d


def rbf_kernel(A, B, gamma=0.1):
    """
    k(x_i, x_j) = exp(-gamma * ||x_i - x_j||^2)
    Efficient computation using the identity:
    ||a - b||^2 = ||a||^2 + ||b||^2 - 2 a·b
    """
    sq_A = np.sum(A ** 2, axis=1, keepdims=True)  # (n, 1)
    sq_B = np.sum(B ** 2, axis=1, keepdims=True)  # (m, 1)
    sq_dist = sq_A + sq_B.T - 2 * (A @ B.T)       # (n, m)
    sq_dist = np.maximum(sq_dist, 0.0)             # numerical safety
    return np.exp(-gamma * sq_dist)


def neural_kernel(phi_A, phi_B):
    """k(x_i, x_j) = φ_NN(x_i) · φ_NN(x_j)"""
    return phi_A @ phi_B.T


print('Kernel functions defined.')

In [ ]:
# ============================================================
# 7.2.2 — TRAIN SVR WITH EACH KERNEL & REPORT SCORES
# ============================================================

print('\n===== SVR Kernel Comparison =====')

# For raw-feature kernels we use the standardised inputs X_train / X_test.
# For the neural kernel we use φ_NN(X_train) / φ_NN(X_test).

kernel_configs = [
    # (label, K_train_fn, K_test_fn)
    (
        'Linear',
        lambda: linear_kernel(X_train, X_train),
        lambda: linear_kernel(X_test, X_train)
    ),
    (
        'Poly d=2',
        lambda: polynomial_kernel(X_train, X_train, d=2),
        lambda: polynomial_kernel(X_test, X_train, d=2)
    ),
    (
        'Poly d=3',
        lambda: polynomial_kernel(X_train, X_train, d=3),
        lambda: polynomial_kernel(X_test, X_train, d=3)
    ),
    (
        'RBF γ=0.01',
        lambda: rbf_kernel(X_train, X_train, gamma=0.01),
        lambda: rbf_kernel(X_test, X_train, gamma=0.01)
    ),
    (
        'RBF γ=0.1',
        lambda: rbf_kernel(X_train, X_train, gamma=0.1),
        lambda: rbf_kernel(X_test, X_train, gamma=0.1)
    ),
    (
        'RBF γ=1.0',
        lambda: rbf_kernel(X_train, X_train, gamma=1.0),
        lambda: rbf_kernel(X_test, X_train, gamma=1.0)
    ),
    (
        'Neural Kernel',
        lambda: neural_kernel(phi_train, phi_train),
        lambda: neural_kernel(phi_test, phi_train)
    ),
]

results = []

for label, K_train_fn, K_test_fn in kernel_configs:

    K_train = K_train_fn()
    K_test  = K_test_fn()

    svr = SVR(kernel='precomputed', C=10.0, epsilon=0.1)
    svr.fit(K_train, y_train)

    y_pred = svr.predict(K_test)

    mse = mean_squared_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)

    results.append({'Kernel': label, 'MSE': mse, 'R2': r2,
                    'K_train': K_train, 'K_test': K_test})

    print(f'{label:15s}  MSE = {mse:8.4f}   R² = {r2:+.4f}')

print('\nDone!')

In [ ]:
# ============================================================
# 7.2.2 — BAR CHART: MSE & R² COMPARISON
# ============================================================

labels = [r['Kernel'] for r in results]
mse_vals = [r['MSE'] for r in results]
r2_vals  = [r['R2']  for r in results]

x = np.arange(len(labels))
width = 0.4

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- MSE ---
bars = axes[0].bar(x, mse_vals, width=0.6, color='steelblue', edgecolor='navy')
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=30, ha='right', fontsize=10)
axes[0].set_ylabel('Mean Squared Error', fontsize=12)
axes[0].set_title('SVR Test MSE by Kernel', fontsize=14)
axes[0].bar_label(bars, fmt='%.2f', fontsize=9, padding=3)
axes[0].grid(axis='y', alpha=0.3)

# --- R² ---
colors = ['green' if v > 0 else 'tomato' for v in r2_vals]
bars2 = axes[1].bar(x, r2_vals, width=0.6, color=colors, edgecolor='darkgreen')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=30, ha='right', fontsize=10)
axes[1].set_ylabel('R² Score', fontsize=12)
axes[1].set_title('SVR Test R² by Kernel', fontsize=14)
axes[1].bar_label(bars2, fmt='%.3f', fontsize=9, padding=3)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 7.2.3 — KERNEL MATRIX VISUALIZATION
# ============================================================

print('\n===== Kernel Matrix Visualizations =====')

# Use a small subset for clear visualisation
n_vis = 200

vis_configs = [
    ('Linear',         linear_kernel(X_train[:n_vis],     X_train[:n_vis])),
    ('Poly d=2',       polynomial_kernel(X_train[:n_vis], X_train[:n_vis], d=2)),
    ('Poly d=3',       polynomial_kernel(X_train[:n_vis], X_train[:n_vis], d=3)),
    ('RBF γ=0.01',     rbf_kernel(X_train[:n_vis],        X_train[:n_vis], gamma=0.01)),
    ('RBF γ=0.1',      rbf_kernel(X_train[:n_vis],        X_train[:n_vis], gamma=0.1)),
    ('RBF γ=1.0',      rbf_kernel(X_train[:n_vis],        X_train[:n_vis], gamma=1.0)),
    ('Neural Kernel',  neural_kernel(phi_train[:n_vis],   phi_train[:n_vis])),
]

ncols = 4
nrows = int(np.ceil(len(vis_configs) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 4))
axes = axes.flatten()

for i, (name, K) in enumerate(vis_configs):
    im = axes[i].imshow(K, aspect='auto', cmap='viridis')
    axes[i].set_title(f'Kernel Matrix\n{name}', fontsize=11)
    axes[i].set_xlabel('Sample index')
    axes[i].set_ylabel('Sample index')
    plt.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)

# Hide unused subplots
for j in range(len(vis_configs), len(axes)):
    axes[j].set_visible(False)

plt.suptitle(f'Kernel Gram Matrices  (first {n_vis} training samples)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 7.2.3 — PCA + SVR DECISION SURFACE VISUALIZATION
# ============================================================

print('\n===== PCA Decision Surface Visualization =====')

def pca_2d(X_fit, X_transform=None):
    """NumPy PCA: fit on X_fit, project X_transform (default: X_fit itself)."""
    if X_transform is None:
        X_transform = X_fit
    mu = X_fit.mean(axis=0)
    Xc = X_fit - mu
    cov = Xc.T @ Xc / len(Xc)
    evals, evecs = np.linalg.eigh(cov)
    # Sort descending
    order = np.argsort(evals)[::-1]
    W = evecs[:, order[:2]]
    return (X_transform - mu) @ W, W, mu


# Project raw features + neural features to 2D
X_train_2d, W_raw, mu_raw = pca_2d(X_train)
X_test_2d                  = (X_test - mu_raw) @ W_raw

phi_train_2d, W_phi, mu_phi = pca_2d(phi_train)
phi_test_2d                  = (phi_test - mu_phi) @ W_phi

# SVR on 2D PCA projections for clean boundary plots
surface_configs = [
    ('Linear (raw)',          X_train_2d,    X_test_2d,    'linear'),
    ('RBF γ=0.1 (raw)',       X_train_2d,    X_test_2d,    'rbf'),
    ('Neural Kernel (φ_NN)',  phi_train_2d,  phi_test_2d,  'linear'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (title, Xtr2, Xte2, kern) in zip(axes, surface_configs):

    svr2d = SVR(kernel=kern, C=10.0, epsilon=0.5)
    if kern == 'rbf':
        svr2d.set_params(gamma=0.1)
    svr2d.fit(Xtr2, y_train)
    preds_2d = svr2d.predict(Xte2)

    # Create grid
    x_min, x_max = Xtr2[:, 0].min() - 0.5, Xtr2[:, 0].max() + 0.5
    y_min, y_max = Xtr2[:, 1].min() - 0.5, Xtr2[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 120),
        np.linspace(y_min, y_max, 120)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = svr2d.predict(grid).reshape(xx.shape)

    cf = ax.contourf(xx, yy, zz, levels=20, cmap='RdYlGn', alpha=0.5)
    sc = ax.scatter(
        Xte2[:, 0], Xte2[:, 1],
        c=y_test, cmap='RdYlGn',
        edgecolors='k', linewidths=0.4,
        s=18, zorder=5
    )
    plt.colorbar(sc, ax=ax, label='mess_duration')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('PCA Component 1')
    ax.set_ylabel('PCA Component 2')
    mse = mean_squared_error(y_test, preds_2d)
    ax.text(0.03, 0.96, f'MSE = {mse:.2f}', transform=ax.transAxes,
            va='top', fontsize=9, bbox=dict(boxstyle='round', fc='white', alpha=0.7))

plt.suptitle('SVR Decision Surfaces in 2D PCA Space', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SUMMARY TABLE
# ============================================================

print('\n' + '='*55)
print(f'{"Kernel":<18}  {"MSE":>10}  {"R²":>10}')
print('='*55)
for r in results:
    print(f"{r['Kernel']:<18}  {r['MSE']:>10.4f}  {r['R2']:>+10.4f}")
print('='*55)

best = min(results, key=lambda r: r['MSE'])
print(f"\nBest kernel: {best['Kernel']}  (MSE = {best['MSE']:.4f}, R² = {best['R2']:+.4f})")

## Observations & Discussion

### Effect of Kernel Choice

| Aspect | What we observe |
|---|---|
| **Linear** | Fastest to compute; baseline. Captures linear correlations between features. |
| **Polynomial d=2/3** | Adds interaction terms; may overfit with very high $d$. |
| **RBF (small γ)** | Smooth, broad influence per support vector → underfitting risk. |
| **RBF (large γ)** | Sharp, localised influence → overfitting risk on test data. |
| **Neural Kernel** | Leverages the learned non-linear task-specific representation $\phi_{NN}(x)$. Often outperforms fixed kernels on tabular data. |

### Neural Kernel Advantage
The neural kernel essentially combines the **representation learning power of a neural network** with the **generalization guarantees of SVMs**. The MLP has already learned how to map the raw 14-dimensional input into a 64-dimensional space that is well-structured for predicting `mess_duration`. The SVR with the neural kernel then fits a maximum-margin regressor in this rich space — giving the best of both worlds.

### Connection to Theory
As shown in Section 7.1, $k_{NN}(x_i, x_j) = \phi_{NN}(x_i)^T \phi_{NN}(x_j)$ is a valid Mercer kernel by construction. The t-SNE visualization of $\phi_{NN}(x)$ demonstrates that the network has learned a smooth manifold over the target space — samples with similar `mess_duration` cluster together in the learned feature space.